# Nano-NLA RL GPU windows

Use this notebook on a GPU Colab runtime. It samples a 20k row window from `rl.parquet`, runs an 800-step bounded GRPO window, and stores each window in its own Drive-backed checkpoint directory. Re-run the RL cell after an interruption; `--resume-latest` loads the newest complete `step_N/av` and `step_N/ar` pair in that window and `--end-step` keeps the step budget bounded.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/nano-nla')
REPO_DIR = Path('/content/Nano-NLA')
REPO_URL = 'https://github.com/IrohAmca/Nano-NLA.git'  # Change this if using a fork.
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if not (REPO_DIR / '.git').exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
import os
os.chdir(REPO_DIR)
print('cwd:', Path.cwd())

In [ ]:
!pip -q install uv
!uv sync

from pathlib import Path

def link_drive_dir(local: Path, persistent: Path) -> None:
    persistent.mkdir(parents=True, exist_ok=True)
    local.parent.mkdir(parents=True, exist_ok=True)
    if local.is_symlink():
        if local.resolve() == persistent.resolve():
            return
        local.unlink()
    elif local.exists():
        raise RuntimeError(f'{local} exists in the Colab checkout; move it before linking Drive artifacts.')
    local.symlink_to(persistent, target_is_directory=True)

link_drive_dir(Path('data/generated'), DRIVE_ROOT / 'data' / 'generated')
link_drive_dir(Path('checkpoints'), DRIVE_ROOT / 'checkpoints')
link_drive_dir(Path('results'), DRIVE_ROOT / 'results')
!nvidia-smi
!uv run python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.version.cuda)"

In [ ]:
from pathlib import Path

ROW_OFFSET = 0
MAX_ROWS = 20000
WINDOW_STEPS = 800

def window_dir_for(offset: int, rows: int) -> Path:
    end_row = offset + rows - 1
    return Path('checkpoints/rl/windows') / f'rows_{offset:06d}_{end_row:06d}'

WINDOW_DIR = window_dir_for(ROW_OFFSET, MAX_ROWS)
LOG_PATH = Path('results/rl_logs') / f'{WINDOW_DIR.name}.log'
if ROW_OFFSET == 0:
    ACTOR_INIT = Path('checkpoints/av_sft')
    CRITIC_INIT = Path('checkpoints/ar_sft')
else:
    PREVIOUS_WINDOW_DIR = window_dir_for(max(0, ROW_OFFSET - MAX_ROWS), MAX_ROWS)
    ACTOR_INIT = PREVIOUS_WINDOW_DIR / 'av'
    CRITIC_INIT = PREVIOUS_WINDOW_DIR / 'ar'

print('row window:      ', ROW_OFFSET, ROW_OFFSET + MAX_ROWS - 1)
print('window steps:    ', WINDOW_STEPS)
print('window dir:      ', WINDOW_DIR)
print('actor init:      ', ACTOR_INIT)
print('critic init:     ', CRITIC_INIT)
print('log path:        ', LOG_PATH)

For the first RL window, run this cell only when warm-start SFT checkpoints are absent. Later windows must use the previous window final `av` and `ar` directories instead.

In [ ]:
if ROW_OFFSET == 0 and (not ACTOR_INIT.exists() or not CRITIC_INIT.exists()):
    !uv run python scripts/run_datagen.py --config configs/qwen05b.yaml --stage 3
    !uv run python scripts/run_sft.py --config configs/qwen05b.yaml --stage both
else:
    print('SFT prerequisite cell skipped.')

In [ ]:
from pathlib import Path
import pyarrow.parquet as pq

RL_DATASET = Path('data/generated/rl.parquet')
assert RL_DATASET.exists(), f'Missing RL dataset: {RL_DATASET}'
assert Path(f'{RL_DATASET}.nla_meta.yaml').exists(), f'Missing RL sidecar for {RL_DATASET}'
assert ACTOR_INIT.exists(), f'Missing actor checkpoint: {ACTOR_INIT}'
assert CRITIC_INIT.exists(), f'Missing critic checkpoint: {CRITIC_INIT}'
assert (ACTOR_INIT / 'nla_meta.yaml').exists(), f'Missing actor sidecar: {ACTOR_INIT}'
assert (CRITIC_INIT / 'nla_meta.yaml').exists(), f'Missing critic sidecar: {CRITIC_INIT}'

rl_rows = pq.ParquetFile(RL_DATASET).metadata.num_rows
assert ROW_OFFSET < rl_rows, f'ROW_OFFSET {ROW_OFFSET} is beyond rl.parquet rows={rl_rows}'
print('rl.parquet rows:', rl_rows)
print('rows available in requested window:', min(MAX_ROWS, rl_rows - ROW_OFFSET))

This cell appends stdout and stderr to a Drive-backed window log. A printed step after the latest saved `step_N` is not a resumable point; resume uses the latest complete checkpoint pair.

In [ ]:
WINDOW_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
!uv run python scripts/run_rl.py \
  --config configs/qwen05b.yaml \
  --dataset data/generated/rl.parquet \
  --actor-checkpoint "{ACTOR_INIT}" \
  --critic-checkpoint "{CRITIC_INIT}" \
  --output-dir "{WINDOW_DIR}" \
  --row-offset {ROW_OFFSET} \
  --max-rows {MAX_ROWS} \
  --end-step {WINDOW_STEPS} \
  --resume-latest 2>&1 | tee -a "{LOG_PATH}"

In [ ]:
from pathlib import Path

def latest_complete_step(window_dir: Path):
    pairs = []
    for path in window_dir.glob('step_*'):
        if not path.is_dir() or not path.name.removeprefix('step_').isdigit():
            continue
        sidecars = ((path / 'av' / 'nla_meta.yaml'), (path / 'ar' / 'nla_meta.yaml'))
        if (path / 'av').is_dir() and (path / 'ar').is_dir() and all(sidecar.is_file() for sidecar in sidecars):
            pairs.append((int(path.name.removeprefix('step_')), path / 'av', path / 'ar'))
    return max(pairs, default=None, key=lambda pair: pair[0])

print('latest complete step pair:', latest_complete_step(WINDOW_DIR))
for final_path in (WINDOW_DIR / 'av', WINDOW_DIR / 'ar'):
    print(final_path, 'exists=', final_path.exists(), 'sidecar=', (final_path / 'nla_meta.yaml').exists())

if LOG_PATH.exists():
    rl_lines = [line for line in LOG_PATH.read_text(encoding='utf-8', errors='replace').splitlines() if '[rl] step=' in line]
    print('\n'.join(rl_lines[-12:]))
else:
    print('No window log exists yet:', LOG_PATH)

Optional checkpoint checks after a completed window. Keep these samples small inside Colab, then run broader evaluation separately when the checkpoint is worth comparing.

In [ ]:
!uv run python scripts/run_eval.py \
  --config configs/qwen05b.yaml \
  --mode steganography \
  --av-checkpoint "{WINDOW_DIR / 'av'}" \
  --ar-checkpoint "{WINDOW_DIR / 'ar'}" \
  --gold-parquet data/generated/rl.parquet \
  --max-rows 8 \
  --output "{WINDOW_DIR / 'eval_steganography_small.json'}"

In [ ]:
!uv run python scripts/run_eval.py \
  --config configs/qwen05b.yaml \
  --mode confabulation \
  --av-checkpoint "{WINDOW_DIR / 'av'}" \
  --ar-checkpoint "{WINDOW_DIR / 'ar'}" \
  --gold-parquet data/generated/rl.parquet \
  --max-rows 8 \
  --output "{WINDOW_DIR / 'eval_confabulation_small.json'}"